# as-strided-windowing — faded example 3: Batched strided 1-D windows for conv1d prep (complete the stride tuple)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `as-strided-windowing`. Running the beacon reports progress on the `PyTorch: as_strided windowing` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: as_strided windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`as-strided-windowing`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-windowing"
DD_SUBTOPIC = "PyTorch: as_strided windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Extending strided 1-D windowing to a batch of `(B, W)` signals gives a `(B, L_out, K)` view. The batch stride `s_B` is preserved, while the windowing axes use `(s_W * step, s_W)` — outer jumps `step` spatial elements per window, inner walks within a window. No convolution is performed; this is only the input-prep view.

## Faded exercise 3

### Faded — batched strided windows via as_strided

Implement `batched_strided_windows(x, K, step)` for an input of shape `(B, W)`. Return a zero-copy `(B, L_out, K)` view where `L_out = (W - K) // step + 1` and, for each batch `b`, row `i` is `x[b, i*step : i*step + K]`. The shape and `as_strided` call are written. **You must complete the 3-axis `stride` tuple.**

**Fill in:** The 3-axis stride tuple `(s_B, s_W * step, s_W)`: batch stride, the strided window-origin step, and the within-window step.

In [ ]:
def batched_strided_windows(x: Tensor, K: int, step: int) -> Tensor:
    B, W = x.shape
    s_B, s_W = x.stride()
    L_out = (W - K) // step + 1
    stride = (s_B, s_W * step, s_W)
    return t.as_strided(x, size=(B, L_out, K), stride=stride)


def _test():
    t.manual_seed(0)
    x = t.randn(4, 11)
    for K, step in [(3, 1), (3, 2), (4, 2)]:
        out = batched_strided_windows(x, K, step)
        L_out = (x.shape[1] - K) // step + 1
        assert out.shape == (4, L_out, K), out.shape
        for b in range(4):
            for i in range(L_out):
                assert t.equal(out[b, i], x[b, i * step:i * step + K]), (b, i)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def batched_strided_windows(x: Tensor, K: int, step: int) -> Tensor:
    B, W = x.shape
    s_B, s_W = x.stride()
    L_out = (W - K) // step + 1
    stride = (s_B, s_W * step, s_W)
    return t.as_strided(x, size=(B, L_out, K), stride=stride)
```
</details>